# 07 - GAEZ Suitability: Switch All 4 Crops to the Irrigated Scenario

`download_gaez.ipynb`'s original config used `HRLM` (rain-fed, high input) for all four crops,
explicitly "kept the same across every crop/variable so results stay comparable." A first pass
(see git history / the `*_rainfed` audit columns) switched **rice only** to `HILM` (irrigated,
high input), since wetland/paddy rice is conventionally irrigated and the rain-fed-only version
understated it almost everywhere - confirmed by local knowledge (rice is extensive in north/
central Banten) and by the fact rice's score rose most exactly in the zone matching that claim.

That rice-only switch broke the original comparability principle: rice was now being compared
against maize/cassava/soybean still sitting on the *worse* rain-fed scenario, which isn't a fair
basis for picking a "best crop" either. This notebook extends the same `HILM` switch to the
remaining three crops, restoring a consistent scenario across all four - same intent as the
original download, corrected scenario. Old rain-fed values are kept as `*_rainfed` audit columns
for every crop (not just rice). Re-running this notebook is safe: a crop that already has its
`_rainfed` audit column is skipped.

In [1]:
# add scripts/download to sys.path so we can `from common import ...`
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "preprocessing":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

print("cwd:", os.getcwd())

cwd: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\scripts\preprocessing


## 1. Download suitability under the irrigated scenario (HILM), all 4 crops

In [2]:
from common import RAW_DIR, PROCESSED_DIR, ensure_dir, get_logger, download_file

log = get_logger("fix_gaez_irrigated")

BASE_URL_TMPL = "https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPSET/{variable}/"
GAEZ_DIR = RAW_DIR / "gaez_v5"
ensure_dir(GAEZ_DIR)

PERIOD = "HP0120"
CLIMATE_SOURCE = "AGERA5"
SSP = "HIST"
WATER_INPUT_IRRIGATED = "HILM"  # irrigation, high input level of management

CROP_CODES = {"rice": "RCW", "maize": "MZE", "cassava": "CSV", "soybean": "SOY"}
VARIABLES = {"RES05-SIX": "suitability_class", "RES05-SXX": "suitability_index"}

irrigated_paths = {}  # (crop, variable_label) -> Path
for crop, code in CROP_CODES.items():
    for variable, label in VARIABLES.items():
        dest = GAEZ_DIR / f"{label}_{crop}_{code}_irrigated.tif"
        fname = f"GAEZ-V5.{variable}.{PERIOD}.{CLIMATE_SOURCE}.{SSP}.{code}.{WATER_INPUT_IRRIGATED}.tif"
        url = BASE_URL_TMPL.format(variable=variable) + fname
        download_file(url, dest)
        irrigated_paths[(crop, label)] = dest

print("Done -", len(irrigated_paths), "irrigated rasters ready in", GAEZ_DIR)

[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_class_rice_RCW_irrigated.tif


[download] Already have it, skipping: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_index_rice_RCW_irrigated.tif


suitability_class_maize_MZE_irrigated.tif:   0%|          | 0.00/695k [00:00<?, ?B/s]

suitability_class_maize_MZE_irrigated.tif:   9%|▉         | 65.5k/695k [00:00<00:07, 85.3kB/s]

suitability_class_maize_MZE_irrigated.tif:  19%|█▉        | 131k/695k [00:01<00:05, 106kB/s]  

suitability_class_maize_MZE_irrigated.tif:  28%|██▊       | 197k/695k [00:01<00:03, 140kB/s]

... (78 outputs left out)

suitability_index_soybean_SOY_irrigated.tif:  82%|████████▏ | 3.15M/3.82M [00:03<00:00, 3.74MB/s]

suitability_index_soybean_SOY_irrigated.tif: 100%|██████████| 3.82M/3.82M [00:03<00:00, 1.20MB/s]


[download] Done: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5\suitability_index_soybean_SOY_irrigated.tif


Done - 8 irrigated rasters ready in D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\raw\gaez_v5


## 2. Zonal statistics onto the existing 1 km grid (same method as notebook 04)

In [3]:
import logging

import geopandas as gpd
from rasterstats import zonal_stats

logging.getLogger("rasterio._env").setLevel(logging.ERROR)  # silence repeated GDAL_DATA noise, not an error

GRID_PATH = PROCESSED_DIR / "grid" / "grid_1km.gpkg"
grid = gpd.read_file(GRID_PATH)
grid_4326 = grid.to_crs("EPSG:4326")
print(f"{len(grid)} grid cells loaded")

# all_touched=True: GAEZ's ~9 km pixels are much coarser than our 1 km cells (see notebook 04).
irrigated_values = {}  # (crop, "class"|"index") -> list of values, one per grid cell
for crop in CROP_CODES:
    class_stats = zonal_stats(
        grid_4326.geometry, str(irrigated_paths[(crop, "suitability_class")]),
        stats=["majority"], all_touched=True,
    )
    index_stats = zonal_stats(
        grid_4326.geometry, str(irrigated_paths[(crop, "suitability_index")]),
        stats=["mean"], all_touched=True,
    )
    irrigated_values[(crop, "class")] = [s["majority"] for s in class_stats]
    irrigated_values[(crop, "index")] = [s["mean"] for s in index_stats]
    n_valid = sum(v is not None for v in irrigated_values[(crop, "index")])
    print(f"{crop}: {n_valid}/{len(grid)} cells got a value")

D:\Softwares\Anaconda\Miniconda\envs\geoai-suitability\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
D:\Softwares\Anaconda\Miniconda\envs\geoai-suitability\Lib\site-packages\rasterio\env.py:664: RuntimeWarning: Cannot find gdalvrt.xsd (GDAL_DATA is not defined)
  elif GDALDataFinder().find_file("gdalvrt.xsd"):


9333 grid cells loaded


rice: 9333/9333 cells got a value


maize: 9333/9333 cells got a value


cassava: 9333/9333 cells got a value


soybean: 9333/9333 cells got a value


## 3. Patch `tabular_features.csv` - skip any crop already fixed (idempotent)

In [4]:
import pandas as pd

FEATURES_PATH = PROCESSED_DIR / "features" / "tabular_features.csv"
df = pd.read_csv(FEATURES_PATH)
assert len(df) == len(grid), "row count mismatch between features table and grid"

changed = []
for crop in CROP_CODES:
    rainfed_col = f"gaez_index_{crop}_rainfed"
    if rainfed_col in df.columns:
        print(f"{crop}: already on the irrigated scenario (audit column present) - skipping")
        continue
    # preserve the original rain-fed values before overwriting
    df[f"gaez_class_{crop}_rainfed"] = df[f"gaez_class_{crop}"]
    df[f"gaez_index_{crop}_rainfed"] = df[f"gaez_index_{crop}"]
    df[f"gaez_class_{crop}"] = irrigated_values[(crop, "class")]
    df[f"gaez_index_{crop}"] = irrigated_values[(crop, "index")]
    changed.append(crop)

df.to_csv(FEATURES_PATH, index=False)
print(f"Wrote {FEATURES_PATH} - {len(df)} rows, {len(df.columns)} columns")
print("Crops patched this run:", changed)

rice: already on the irrigated scenario (audit column present) - skipping


Wrote D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\processed\features\tabular_features.csv - 9333 rows, 45 columns
Crops patched this run: ['maize', 'cassava', 'soybean']


## 4. Validate: full label distribution now that all 4 crops are on the same scenario

In [5]:
import numpy as np

CROPS = ["rice", "maize", "cassava", "soybean"]
NOT_RECOMMENDED_THRESHOLD = 5000
TIE_THRESHOLD = 500


def label_cells(frame):
    idx_cols = [f"gaez_index_{c}" for c in CROPS]
    scores = frame[idx_cols].to_numpy()
    order = np.argsort(scores, axis=1)
    best_i, second_i = order[:, -1], order[:, -2]
    best_score = scores[np.arange(len(frame)), best_i]
    second_score = scores[np.arange(len(frame)), second_i]
    names = np.array([c.capitalize() for c in CROPS])
    labels = names[best_i].astype(object)
    tie_mask = (best_score - second_score) <= TIE_THRESHOLD
    labels[tie_mask] = [
        " / ".join(sorted((names[b], names[s]))) + " (tied)"
        for b, s in zip(best_i[tie_mask], second_i[tie_mask])
    ]
    labels[best_score < NOT_RECOMMENDED_THRESHOLD] = "Not recommended"
    return pd.Series(labels, index=frame.index)


new_label = label_cells(df)
print("New distribution (all 4 crops irrigated):")
print((new_label.value_counts(normalize=True) * 100).round(1))

lat_tertiles = pd.qcut(df["lat"], 3, labels=["South (upland)", "Central", "North (coastal)"])
print()
print("Mean suitability index per crop, by zone (irrigated scenario):")
print(df.groupby(lat_tertiles, observed=True)[[f"gaez_index_{c}" for c in CROPS]].mean().round(0))

print()
print("Solo-winner counts:")
solo_mask = ~new_label.isin(["Not recommended"]) & ~new_label.str.contains(" / ", na=False)
print(new_label[solo_mask].value_counts())

New distribution (all 4 crops irrigated):
Not recommended          66.2
Cassava                  21.5
Cassava / Rice (tied)    12.3
Name: proportion, dtype: float64

Mean suitability index per crop, by zone (irrigated scenario):
                 gaez_index_rice  gaez_index_maize  gaez_index_cassava  \
lat                                                                      
South (upland)            1784.0             836.0              2645.0   
Central                   2683.0            1129.0              3921.0   
North (coastal)           4650.0            1978.0              5313.0   

                 gaez_index_soybean  
lat                                  
South (upland)               1005.0  
Central                      1658.0  
North (coastal)              2542.0  

Solo-winner counts:
Cassava    2003
Name: count, dtype: int64


## Notes

- All 4 crops now use `HILM` (irrigated, high input) - restores the original download notebook's
  "same scenario for every crop" comparability principle, with the scenario itself corrected.
- `gaez_class_<crop>_rainfed` / `gaez_index_<crop>_rainfed` preserve the original `HRLM` values for
  every crop, for comparison/audit - not used by the labeling rule.
- Does **not** touch Assignment I's written report, slide deck, or presentation script - those keep
  documenting the original all-rain-fed result as a historical snapshot, per instruction.
- The dashboard reads `tabular_features.csv` directly, so it reflects this automatically next run.